# LEARN HOW A TRANSFORMER OR LLM IS DESIGNED!

## Dataset Formation & Design

**DATASET DOWNLOADING IN CHUNKS FOR JUST EDU DOMAIN**                                       
 (COMMON CRAWL = RAW DATASET ON WEB JUNK)..........                                       what we used is = **(Fine Web 1 .edu 100M sample data with _ tokens for a small transformer design!!)**

In [ ]:
from datasets import load_dataset
import os

# 1. FineWeb-Edu ka 10 Billion tokens wala sample load karein
# Hum sirf 'train' split ka aik chota sa hissa (streaming=True) use karenge

ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

# 2. Sirf pehla 10,000 documents save karte hain (shuruat ke liye kaafi hain)

print("Data download ho raha hai...")
with open("my_data.txt", "w", encoding="utf-8") as f:
    for i, entry in enumerate(ds):
        f.write(entry['text'] + "\n\n")
        if i >= 10000:           # 10k rows ke baad ruk jao
            break

print("Done! 'my_data.txt' file ban gayi hai.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Data download ho raha hai...
Done! 'my_data.txt' file ban gayi hai.


In [ ]:
# Pehli 500 characters print karein

with open("my_data.txt", "r") as f:
    print(f.read(500))

The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath


In [ ]:
from google.colab import files
files.download('my_data.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# After mounting, you can access your file at:
# /content/drive/MyDrive/Transformer_to_LLM/my_data.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Processing data in standard numeric file - 'safetensors'...**

In [ ]:
import tiktoken
import torch
from safetensors.torch import save_file    # safetensors library se save_file
import numpy as np

# 1. GPT-2 ka tokenizer load karein

enc = tiktoken.get_encoding("gpt2")

# 2. Apni file read karein

with open("/content/drive/MyDrive/Transformer_to_LLM/my_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 3. Text ko tokens (numbers) mein convert karein

tokens = enc.encode(text)
print(f"Total tokens: {len(tokens)}")

# 4. In tokens ko numpy array mein badlein (dtype uint16 sahi hai)

tokens_np = np.array(tokens, dtype=np.uint16)

# 5. --- CHANGE YAHAN HAI ---

# Numpy tokens ko torch tensor mein convert karein (int32 ya long/int64 chahiye)
# Safetensors mein save karne ke liye tensor ka type int32 hona behtar hai

tokens_tensor = torch.from_numpy(tokens_np.astype(np.int32))

# 6. Safetensors file save karein

save_file({"input_ids": tokens_tensor}, "train_tokens.safetensors")
print("Tokens save ho gaye: train_tokens.safetensors")

Total tokens: 10324699
Tokens save ho gaye: train_tokens.safetensors


In [ ]:
from google.colab import files
files.download('train_tokens.safeten)

SyntaxError: unterminated string literal (detected at line 2) (728128642.py, line 2)

**Setting dimensions and size - MODEL CONFIGURATION!**

In [ ]:
import torch

# Model ki settings defined variables used... (Configuration)
# Updated Settings for a "Solid" Student Model

vocab_size = 50257      # GPT-2 ki vocabulary ka size
batch_size= 32
n_layer = 6             # Deep enough to be smart
n_head = 6              # # 256 / 8 = 32 (Perfect calculation) (har head 32 dim ka hoga, perfect kam capacity ma)
n_embd = 384            # A good middle ground for 10M tokens
block_size = 256        # Can read half a page at once
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Transformer Architecture Core

**Step 1: Self-Attention Block (The Focus)**            
Ye layer model ko sikhati hai ke sentence mein kaunsa word kis se juda hai...

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
    """ Ek single attention head """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        # Attention scores (affinities) calculate karna
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads parallel mein kaam karte hain """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out


**Step 2: Feed-Forward Network (The Memory)**.............................................................               Jab model attention se relation samajh leta hai, toh ye layer usko "process" karti hai

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)


**Step 3: The Transformer Block & Full Model**...............................................................
Ab hum sab ko mila kar aik "Block" banayenge aur usko 6 baar repeat karenge (layers).

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # Residual connections
        x = x + self.ffwd(self.ln2(x))
        return x

class MyStudentGPT(nn.Module):
    def __init__(self):
        super().__init__()
        # Token and Position Embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # Apply 6 layers of Transformer blocks
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T).long() # Convert targets to long
            loss = F.cross_entropy(logits, targets)

        return logits, loss

*Ab hum Training Loop likhenge. Ye wo stage hai jahan aapka banaya hua "Brain" (Model) aapki di hui "Book" (FineWeb Data) ko parhna shuru karega.*

## Training Loop (Ready the Brain of Model)

**Step 1: Data Loader:**....................................... Jo .safetensors se data nikaal kar model ko chote chote tukron (batches) mein de.

In [ ]:
from safetensors.torch import load_file

# 1. Safetensors file se tokens wapas load karein

data_dict = load_file("train_tokens.safetensors")
data = data_dict["input_ids"]               # Ye hamare saare numbers hain

def get_batch():
    """
    Data mein se random tukre (chunks) uthane ke liye.
    B = Batch Size (Ek sath kitne sentences process honge)
    T = Block Size (Har sentence kitna lamba hoga)
    """
    ix = torch.randint(len(data) - block_size, (batch_size,))         # Random starting points
    x = torch.stack([data[i:i+block_size] for i in ix])               # Input (Sawal)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])           # Target (Sahi Jawab - agla word)

    # Data ko GPU par bhejna (agar available ho)

    x, y = x.to(device), y.to(device)
    return x, y

# Training Hyperparameters (Setings)

batch_size = 32            # Ek bar mein 8 lines process hongi (Further reduced to save memory)
max_iters = 7000           # Model kitni baar practice karega
learning_rate = 5e-4      # Model kitni tezi se seekhega (zyada fast ho to confuse ho jata hai)
eval_interval = 100        # Har 100 steps ke baad check karenge model kaisa perform kar raha hai

**Step 2: Main Training of my Model- 'with Loss minimization**

**Recommended Naming Structure: [ChannelName]-[Size]-[Variant]**


### Fluffy_AI_Desk_base_v1.0 (small)

In [ ]:
# Model ko initialize karna
model = MyStudentGPT()
m = model.to(device) # Model ko GPU (T4) par transfer karein

# Optimizer: Ye model ki "Ghaltiyan" (Loss) kam karne mein madad karta hai
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

print("Training shuru ho rahi hai...")

for iter in range(max_iters):

    # 1. Data ka ek batch uthao
    xb, yb = get_batch()

    # 2. Model ko data dikhao aur "Loss" (ghalti) calculate karo
    logits, loss = model(xb, yb)

    # 3. Purani ghaltiyon ko saaf karo
    optimizer.zero_grad(set_to_none=True);

    # 4. "Backpropagation": Dekho kaunse weights ne ghalti ki
    loss.backward()

    # 5. Weights ko update karo (Improvement)
    optimizer.step()

    # Har thori der baad status print karo
    if iter % eval_interval == 0:
        print(f"Step {iter}: Loss calculate ho raha hai...")
        print(f"Iter {iter}: Loss {loss.item():.4f}")

print("Training Mukammal!")

Training shuru ho rahi hai...
Step 0: Loss calculate ho raha hai...
Iter 0: Loss 10.9796


KeyboardInterrupt: 

In [ ]:
# Inference (Generation) ka code
model.eval() # Model ko testing mode mein dalen
start_context = torch.zeros((1, 1), dtype=torch.long, device=device) # Empty starting point

# Generate function (aapke model class mein ye function hona chahiye, ya niche wala generic use karen)
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        # Crop idx to the last block_size tokens
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] # Last time step par focus karen
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

# Output dekhein
generated_tokens = generate(model, start_context, max_new_tokens=100)[0].tolist()
print(enc.decode(generated_tokens))


!D Prize, 219if self research as well.
groups lower example Press, rotated is their children as they fadeed and observes a free through herism, and politics:
Also your 1911,
It was at the magazine thought of port a close. Only
doi last vote of the doors in AM speech campaigns against 12.
Over a cabin. This, the moon is thought to night notes up your own Tugs on your sources:
"histon will be just reserved inappropriate out


In [ ]:
import gc
import torch

# GPU memory clear karne ke liye
gc.collect()
torch.cuda.empty_cache()
